# SIR Epidemic Model — ML Parameter Learning
> **Author:** Islam Mahmoud | [@IslamMahmoud-ai](https://github.com/IslamMahmoud-ai)

This notebook explores the SIR epidemic model and trains ML models to predict epidemic outcomes from model parameters.

### What is the SIR Model?
- **S** = Susceptible (can get infected)
- **I** = Infected (currently sick)
- **R** = Removed (recovered or deceased)

The model is defined by two parameters:
- **β (beta)** = transmission rate
- **γ (gamma)** = recovery rate
- **R₀ = β/γ** = basic reproduction number (R₀ > 1 → epidemic spreads)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
print('Libraries loaded ✓')

## 1. Simulate a Single SIR Epidemic

In [ ]:
def sir_equations(y, t, beta, gamma, N):
    S, I, R = y
    dS = -beta * S * I / N
    dI =  beta * S * I / N - gamma * I
    dR =  gamma * I
    return dS, dI, dR

def simulate_sir(beta, gamma, N=10_000, I0=10, days=160):
    S0 = N - I0
    t = np.linspace(0, days, days)
    sol = odeint(sir_equations, (S0, I0, 0), t, args=(beta, gamma, N))
    S, I, R = sol.T
    return pd.DataFrame({'t': t, 'S': S, 'I': I, 'R': R})

# Simulate with beta=0.3, gamma=0.1 → R₀ = 3
df = simulate_sir(beta=0.3, gamma=0.1)

plt.plot(df['t'], df['S'], label='Susceptible', color='#3B8BD4', lw=2)
plt.plot(df['t'], df['I'], label='Infected',    color='#E24B4A', lw=2)
plt.plot(df['t'], df['R'], label='Removed',     color='#1D9E75', lw=2)
plt.xlabel('Days')
plt.ylabel('Population')
plt.title('SIR Epidemic Curve  (β=0.3, γ=0.1, R₀=3.0)')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Peak infected: {df['I'].max():.0f} on day {df['I'].idxmax()}")

## 2. Effect of R₀ on Epidemic Spread

In [ ]:
scenarios = [
    {'beta': 0.15, 'gamma': 0.1, 'label': 'R₀=1.5 (slow spread)'},
    {'beta': 0.30, 'gamma': 0.1, 'label': 'R₀=3.0 (moderate)'},
    {'beta': 0.50, 'gamma': 0.1, 'label': 'R₀=5.0 (fast spread)'},
]
colors = ['#3B8BD4', '#E24B4A', '#BA7517']

for scenario, color in zip(scenarios, colors):
    df = simulate_sir(scenario['beta'], scenario['gamma'])
    plt.plot(df['t'], df['I'], label=scenario['label'], color=color, lw=2)

plt.xlabel('Days')
plt.ylabel('Infected population')
plt.title('Effect of R₀ on Epidemic Spread')
plt.legend()
plt.tight_layout()
plt.show()

## 3. Generate Synthetic Dataset (500 Epidemics)

In [ ]:
def generate_dataset(n_samples=500, N=10_000, seed=42):
    rng = np.random.default_rng(seed)
    betas  = rng.uniform(0.1, 0.5, n_samples)
    gammas = rng.uniform(0.05, 0.3, n_samples)
    records = []
    for beta, gamma in zip(betas, gammas):
        df = simulate_sir(beta, gamma, N=N)
        records.append({
            'beta': beta,
            'gamma': gamma,
            'R0': beta / gamma,
            'peak_infected': df['I'].max(),
            'peak_day': df['I'].idxmax(),
            'final_removed': df['R'].iloc[-1],
        })
    return pd.DataFrame(records)

data = generate_dataset(500)
print(f'Dataset shape: {data.shape}')
data.head()

In [ ]:
# Distribution of R₀ values in our dataset
plt.hist(data['R0'], bins=30, color='#3B8BD4', edgecolor='white')
plt.axvline(x=1, color='#E24B4A', linestyle='--', label='R₀=1 (epidemic threshold)')
plt.xlabel('R₀ (Basic Reproduction Number)')
plt.ylabel('Count')
plt.title('Distribution of R₀ in Synthetic Dataset')
plt.legend()
plt.tight_layout()
plt.show()

## 4. Train ML Model — Predict Epidemic Outcomes

In [ ]:
features = ['beta', 'gamma', 'R0']
targets  = ['peak_infected', 'peak_day', 'final_removed']
X = data[features].values

print(f"{'Target':<22} {'RMSE':>10} {'R² Score':>10}")
print('-' * 44)

all_results = {}
for target in targets:
    y = data[target].values
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)
    all_results[target] = {'y_test': y_test, 'y_pred': y_pred, 'r2': r2, 'rmse': rmse}
    print(f"{target:<22} {rmse:>10.2f} {r2:>10.4f}")

## 5. Actual vs Predicted

In [ ]:
colors = ['#3B8BD4', '#E24B4A', '#1D9E75']
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, target, color in zip(axes, targets, colors):
    y_test = all_results[target]['y_test']
    y_pred = all_results[target]['y_pred']
    r2     = all_results[target]['r2']
    ax.scatter(y_test, y_pred, alpha=0.4, color=color, s=20)
    lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    ax.plot(lims, lims, 'k--', lw=1)
    ax.set_xlabel('Actual')
    ax.set_ylabel('Predicted')
    ax.set_title(f'{target}\nR² = {r2:.4f}')

plt.suptitle('ML Model: Actual vs Predicted', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Feature Importance

In [ ]:
# Retrain on peak_infected to get feature importances
y = data['peak_infected'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

importances = model.feature_importances_
plt.barh(features, importances, color=['#3B8BD4', '#E24B4A', '#1D9E75'])
plt.xlabel('Feature Importance')
plt.title('Feature Importance — Predicting Peak Infections')
plt.tight_layout()
plt.show()

## Summary

| What we did | Result |
|---|---|
| Simulated 500 SIR epidemics | ✅ |
| Trained Random Forest ML model | ✅ |
| Predicted peak infections | R² ≈ 0.99 |
| Predicted peak day | R² ≈ 0.97 |
| Predicted final removed | R² ≈ 0.99 |

**Next steps:**
- Try PyTorch neural network instead of Random Forest
- Add auto-differentiation to recover differential equations symbolically
- Extend to SEIR model (adds Exposed compartment)